In [3]:
# ── Standard library ───────────────────────────────────────────────────────
import os
import re
from pathlib import Path
from datetime import datetime

# ── Data science ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Geospatial ─────────────────────────────────────────────────────────────
from geopy.distance import geodesic

# ── Display settings ────────────────────────────────────────────────────────
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
plt.style.use("dark_background")

# ──────────────────────────────────────────────────────────────────────────
# PROJECT ROOT — pathlib makes paths OS-agnostic (works on Windows AND Linux)
# Path(__file__) would be the script path, but in notebooks we use Path.cwd()
# ──────────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path("C:/xcas-ga-comms-assistant")  # Change if different

# ── Data paths ──────────────────────────────────────────────────────────────
ADSB_PATH    = PROJECT_ROOT / "tartan_data/kbtp/raw/2020/10-22-20/1.csv"
AUDIO_DIR    = PROJECT_ROOT / "tartan_data/kbtp/2020/10/10-22-20_audio"
WEATHER_PATH = PROJECT_ROOT / "tartan_data/weather/BTP.csv"

# ── Output paths ────────────────────────────────────────────────────────────
INTERIM_DIR  = PROJECT_ROOT / "data/interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# ── Airport anchor (KBTP — Butler County Regional Airport, PA) ──────────────
KBTP_LAT  = 40.7769
KBTP_LON  = -79.9697
KBTP_ELEV = 1248   # feet MSL
KBTP_NAME = "Butler"   # Used in radio callouts: "Butler Traffic"

print("✅ Paths configured")
print(f"   ADS-B  : {ADSB_PATH}")
print(f"   Audio  : {AUDIO_DIR}")
print(f"   Weather: {WEATHER_PATH}")

✅ Paths configured
   ADS-B  : C:\xcas-ga-comms-assistant\tartan_data\kbtp\raw\2020\10-22-20\1.csv
   Audio  : C:\xcas-ga-comms-assistant\tartan_data\kbtp\2020\10\10-22-20_audio
   Weather: C:\xcas-ga-comms-assistant\tartan_data\weather\BTP.csv


In [4]:
# ── Load ADS-B CSV ───────────────────────────────────────────────────────────
# The raw data has list-formatted strings for Time and Date columns
# e.g., Time = "[u'08', u'20', u'08.935']" — we'll parse these properly

df_raw = pd.read_csv(ADSB_PATH)

print("═" * 60)
print("ADS-B RAW DATA PROFILE")
print("═" * 60)
print(f"Shape          : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"\nColumn names   : {list(df_raw.columns)}")
print(f"\nData types:\n{df_raw.dtypes}")
print(f"\nNull counts:\n{df_raw.isnull().sum()}")
print(f"\nFirst 3 rows:")
display(df_raw.head(3))

════════════════════════════════════════════════════════════
ADS-B RAW DATA PROFILE
════════════════════════════════════════════════════════════
Shape          : 233,868 rows × 12 columns

Column names   : ['ID', 'Time', 'Date', 'Altitude', 'Speed', 'Heading', 'Lat', 'Lon', 'Age', 'Range', 'Bearing', 'Tail']

Data types:
ID          float64
Time            str
Date            str
Altitude    float64
Speed       float64
Heading     float64
Lat         float64
Lon         float64
Age         float64
Range       float64
Bearing     float64
Tail            str
dtype: object

Null counts:
ID           722
Time         722
Date         722
Altitude     722
Speed       1923
Heading     1924
Lat          723
Lon          723
Age          723
Range        723
Bearing      723
Tail        1547
dtype: int64

First 3 rows:


,ID,Time,Date,Altitude,Speed,Heading,Lat,Lon,Age,Range,Bearing,Tail
0,11337771.0000,"[u'08', u'20', u'08.935']","[u'2020', u'10', u'22']",9400.0000,263.0000,268.0000,40.5121,-79.6006,2.7063,41.7557,134.8587,FDX1986
1,11337771.0000,"[u'08', u'20', u'11.646']","[u'2020', u'10', u'22']",9400.0000,263.0000,268.0000,40.5121,-79.6006,0.7098,41.7557,134.8587,FDX1986
2,11337771.0000,"[u'08', u'20', u'11.866']","[u'2020', u'10', u'22']",9400.0000,263.0000,268.0000,40.5121,-79.6006,1.7115,41.7557,134.8587,FDX1986
